In [ ]:
"""
methods.py  (dynamic / vehicular version)

Each method runs over T_horizon time slots.
Cars move between slots — channels are regenerated each slot.
The beamformer warm-starts from the previous slot's solution
(tracking mode) to save iterations.
"""

import time
import numpy as np
from scipy.optimize import minimize

from simulator import (
    generate_channels_at_slot, effective_channels,
    compute_sum_rate, init_beamformers, project_power,
    phase_choices_to_coeffs, PHASE_LEVELS_2BIT,
)
from hamiltonian import (
    build_pauli_hamiltonian, expectation_value, analytic_gradient,
)

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
import scipy.linalg as la


# ── shared: one-slot gradient update ─────────────────────────────────────────

def _gradient_update_slot(H_BR, H_r, H_t, p, W, phases,
                           lr_w=0.05, lr_phi=0.03, n_iter=10):
    """
    Run n_iter gradient steps on (W, phases) for a single channel slot.
    Returns updated (W, phases, R_final, history).
    Warm-starts from supplied W and phases so it tracks across slots.
    """
    N, Nt = p['N'], p['Nt']
    K     = p['Kr'] + p['Kt']
    eps   = 1e-4
    hist  = []

    for _ in range(n_iter):
        beta  = np.sqrt(0.5) * np.exp(1j * phases)
        h     = effective_channels(H_BR, H_r, H_t, beta, beta)
        R     = compute_sum_rate(h, W, p['sigma2'])
        hist.append(R)

        # gradient on W
        gW = np.zeros_like(W)
        for i in range(Nt):
            for j in range(K):
                for part in (1, 1j):
                    W[i,j] += eps * part
                    Rp = compute_sum_rate(
                        effective_channels(H_BR, H_r, H_t, beta, beta),
                        W, p['sigma2'])
                    W[i,j] -= eps * part
                    if part == 1:
                        gW[i,j] += (Rp - R) / eps
                    else:
                        gW[i,j] += 1j * (Rp - R) / eps
        W = project_power(W + lr_w * gW, p['P_max'])

        # gradient on phases
        gp = np.zeros(N)
        for n in range(N):
            phases[n] += eps
            b2 = np.sqrt(0.5) * np.exp(1j * phases)
            Rp = compute_sum_rate(
                effective_channels(H_BR, H_r, H_t, b2, b2), W, p['sigma2'])
            phases[n] -= eps
            gp[n] = (Rp - R) / eps
        phases = (phases + lr_phi * gp) % (2 * np.pi)

    beta   = np.sqrt(0.5) * np.exp(1j * phases)
    h      = effective_channels(H_BR, H_r, H_t, beta, beta)
    R_fin  = compute_sum_rate(h, W, p['sigma2'])
    return W, phases, R_fin, hist


# ═══════════════════════════════════════════════════════════════════════════════
# METHOD 1 — Classical tracking beamformer
# ═══════════════════════════════════════════════════════════════════════════════

def method_classical(cars, p, max_iter_per_slot=20):
    """
    At each slot:
      - move cars
      - regenerate channels
      - run gradient steps (warm-started from previous slot)
    Returns per-slot metrics averaged and as time series.
    """
    t0   = time.time()
    N, Nt, K = p['N'], p['Nt'], p['Kr'] + p['Kt']
    T    = p['T_horizon']
    c1   = 1.0

    W      = init_beamformers(Nt, K, p['P_max'])
    phases = np.random.uniform(0, 2*np.pi, N)
    sr_ts  = []   # sum-rate time series
    iter_ts = []

    for t in range(T):
        # move cars
        for car in cars:
            car.step(p['T_slot'])

        H_BR, H_r, H_t = generate_channels_at_slot(cars, p)
        W, phases, R, hist = _gradient_update_slot(
            H_BR, H_r, H_t, p, W, phases,
            n_iter=max_iter_per_slot)

        sr_ts.append(R)
        iter_ts.append(len(hist))

    total_iter = sum(iter_ts)
    return dict(
        sum_rate      = float(np.mean(sr_ts)),
        sum_rate_ts   = sr_ts,
        iterations    = total_iter,
        energy_norm   = c1 * total_iter,
        time_s        = time.time() - t0,
    )


# ═══════════════════════════════════════════════════════════════════════════════
# METHOD 2 — QAOA-style tracking beamformer
# ═══════════════════════════════════════════════════════════════════════════════

def _qaoa_one_slot(H_BR, H_r, H_t, p,
                   gamma_init, beta_init,
                   n_shots=100, max_opt_iter=15):
    """
    Run QAOA variational optimisation for one channel slot.
    Warm-starts from (gamma_init, beta_init) passed in from previous slot.
    Returns (best_choices, best_W, best_R, gamma_opt, beta_opt, iters)
    """
    N, Nt, K = p['N'], p['Nt'], p['Kr'] + p['Kt']

    # build Hamiltonian coefficients from current channel
    beta_r = phase_choices_to_coeffs(np.zeros(N, int))
    h_ref  = effective_channels(H_BR, H_r, H_t, beta_r, beta_r)
    W_ref  = init_beamformers(Nt, K, p['P_max'])
    hn     = np.mean(np.abs(h_ref)**2, axis=1)
    a = float(np.sum(hn[:K//2]))
    b = float(np.sum(hn[K//2:]))
    c = float(np.mean(np.abs(h_ref))**2 * N)

    iters_used = [0]

    def obj(params):
        iters_used[0] += 1
        return expectation_value(params[0], params[1], a, b, c)

    def jac(params):
        dg, db = analytic_gradient(params[0], params[1], a, b, c)
        return np.array([dg, db])

    res = minimize(obj, [gamma_init, beta_init],
                   jac=jac, method='L-BFGS-B',
                   options={'maxiter': max_opt_iter})

    gamma_opt, beta_opt = res.x

    # build and sample QAOA circuit
    H_op = build_pauli_hamiltonian(a, b, c)
    pauli_list = [(str(t.paulis[0]), float(np.real(t.coeffs[0])))
                  for t in H_op]

    qc = QuantumCircuit(2)
    qc.h([0, 1])
    for label, coeff in pauli_list:
        if abs(coeff) < 1e-12:
            continue
        active = [(2 - 1 - i, pp)
                  for i, pp in enumerate(label) if pp != 'I']
        if len(active) == 1:
            q, typ = active[0]
            if typ == 'Z':
                qc.rz(2 * gamma_opt * coeff, q)
        elif len(active) == 2:
            (q0,_), (q1,_) = active
            qc.cx(q0, q1)
            qc.rz(2 * gamma_opt * coeff, q1)
            qc.cx(q0, q1)
    for q in range(2):
        qc.rx(2 * beta_opt, q)
    qc.measure_all()

    sampler = StatevectorSampler()
    counts  = sampler.run([qc], shots=n_shots).result()[0] \
                     .data.meas.get_counts()

    best_R, best_choices, best_W = -np.inf, np.zeros(N, int), W_ref
    for bstr, _ in sorted(counts.items(), key=lambda x: -x[1]):
        x0 = int(bstr[-1]); x1 = int(bstr[-2])
        ch = np.zeros(N, int)
        ch[:N//2] = x0 * 2; ch[N//2:] = x1 * 2
        bc = phase_choices_to_coeffs(ch)
        W_c = init_beamformers(Nt, K, p['P_max'])
        R   = compute_sum_rate(
            effective_channels(H_BR, H_r, H_t, bc, bc),
            W_c, p['sigma2'])
        if R > best_R:
            best_R, best_choices, best_W = R, ch, W_c

    return best_choices, best_W, best_R, gamma_opt, beta_opt, iters_used[0]


def method_qaoa(cars, p, n_shots=100, max_opt_iter=15):
    t0  = time.time()
    T   = p['T_horizon']
    c2  = 1.5
    sr_ts, iter_ts = [], []

    # warm-start QAOA angles
    gamma, beta_q = np.random.uniform(0, np.pi), np.random.uniform(0, np.pi)

    for t in range(T):
        for car in cars:
            car.step(p['T_slot'])
        H_BR, H_r, H_t = generate_channels_at_slot(cars, p)

        _, _, R, gamma, beta_q, it = _qaoa_one_slot(
            H_BR, H_r, H_t, p,
            gamma_init=gamma, beta_init=beta_q,
            n_shots=n_shots, max_opt_iter=max_opt_iter)

        sr_ts.append(R)
        iter_ts.append(it)

    total_iter = sum(iter_ts)
    return dict(
        sum_rate     = float(np.mean(sr_ts)),
        sum_rate_ts  = sr_ts,
        iterations   = total_iter,
        energy_norm  = c2 * total_iter,
        time_s       = time.time() - t0,
    )


# ═══════════════════════════════════════════════════════════════════════════════
# METHOD 3 — Hybrid tracking beamformer
# ═══════════════════════════════════════════════════════════════════════════════

def method_hybrid(cars, p, n_shots=80, max_opt_iter=10, refine_iter=10):
    t0     = time.time()
    T      = p['T_horizon']
    N, Nt  = p['N'], p['Nt']
    K      = p['Kr'] + p['Kt']
    c1, c2 = 1.0, 1.5
    sr_ts, iter_ts_q, iter_ts_c = [], [], []

    gamma, beta_q = np.random.uniform(0, np.pi), np.random.uniform(0, np.pi)
    W      = init_beamformers(Nt, K, p['P_max'])

    for t in range(T):
        for car in cars:
            car.step(p['T_slot'])
        H_BR, H_r, H_t = generate_channels_at_slot(cars, p)

        # step 1: QAOA coarse search
        choices, W_q, R_q, gamma, beta_q, it_q = _qaoa_one_slot(
            H_BR, H_r, H_t, p,
            gamma_init=gamma, beta_init=beta_q,
            n_shots=n_shots, max_opt_iter=max_opt_iter)

        # step 2: classical refinement (warm-start W from QAOA)
        phases = PHASE_LEVELS_2BIT[choices].copy().astype(float)
        W      = W_q.copy()
        W, phases, R_fin, hist = _gradient_update_slot(
            H_BR, H_r, H_t, p, W, phases,
            n_iter=refine_iter)

        sr_ts.append(R_fin)
        iter_ts_q.append(it_q)
        iter_ts_c.append(len(hist))

    iq = sum(iter_ts_q)
    ic = sum(iter_ts_c)
    return dict(
        sum_rate     = float(np.mean(sr_ts)),
        sum_rate_ts  = sr_ts,
        iterations   = iq + ic,
        energy_norm  = c2 * iq + c1 * ic,
        time_s       = time.time() - t0,
    )